# Análisis exploratorio y modelo predictivo: NYC Citi Bike en BigQuery

Este notebook ejecuta, desde Python, las consultas del análisis exploratorio y la evaluación de los modelos
del proyecto. Todo el procesamiento ocurre en BigQuery; el notebook solo recibe resultados agregados.

Para ejecutarlo en Google Colab basta con autenticarse con la cuenta que tiene acceso al proyecto
`bigdata-evaluacion-2-511416`. Las consultas son las mismas de `sql/02_analisis_exploratorio.sql` y
`sql/05_evaluacion_prediccion.sql`.

In [ ]:
# Autenticación (solo necesaria en Google Colab)
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass  # fuera de Colab se usan las credenciales de gcloud

In [ ]:
import matplotlib.pyplot as plt
from google.cloud import bigquery

PROYECTO = "bigdata-evaluacion-2-511416"
TABLA = f"`{PROYECTO}.citibike_optimizada.viajes_particionados`"
PUBLICA = "`bigquery-public-data.new_york_citibike.citibike_trips`"
MODELO_BASE = f"`{PROYECTO}.citibike_optimizada.modelo_prediccion_duracion`"
MODELO_V2 = f"`{PROYECTO}.citibike_optimizada.modelo_prediccion_duracion_v2`"

cliente = bigquery.Client(project=PROYECTO, location="US")

def consulta(sql):
    trabajo = cliente.query(sql)
    df = trabajo.to_dataframe()
    print(f"Bytes procesados: {trabajo.total_bytes_processed / 1e9:.2f} GB")
    return df

## 1. Estadísticos generales de la duración

La diferencia entre el promedio y la mediana muestra la asimetría de la distribución, y el conteo de viajes
sobre 24 horas identifica los registros anómalos.

In [ ]:
estadisticos = consulta(f"""
SELECT
  COUNT(*)                                        AS total_viajes,
  ROUND(AVG(tripduration), 1)                     AS promedio_seg,
  APPROX_QUANTILES(tripduration, 100)[OFFSET(50)] AS mediana_seg,
  MAX(tripduration)                               AS maximo_seg,
  COUNTIF(tripduration > 86400)                   AS viajes_sobre_24h
FROM {TABLA}
""")
estadisticos

## 2. Viajes por hora de inicio y tipo de usuario

Corresponde a la Figura 2 del informe técnico.

In [ ]:
por_hora = consulta(f"""
SELECT
  EXTRACT(HOUR FROM starttime)     AS hora_inicio,
  COUNTIF(usertype = 'Subscriber') AS viajes_abonados,
  COUNTIF(usertype = 'Customer')   AS viajes_ocasionales
FROM {PUBLICA}
WHERE starttime BETWEEN '2016-01-01' AND '2018-05-31'
  AND tripduration BETWEEN 60 AND 7200
GROUP BY hora_inicio
ORDER BY hora_inicio
""")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(por_hora.hora_inicio, por_hora.viajes_abonados / 1e6, marker="o", label="Abonados (Subscriber)")
ax.plot(por_hora.hora_inicio, por_hora.viajes_ocasionales / 1e6, marker="o", label="Ocasionales (Customer)")
ax.set_xlabel("Hora de inicio")
ax.set_ylabel("Viajes (millones)")
ax.set_xticks(range(24))
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Viajes por día de la semana

En BigQuery, `DAYOFWEEK` toma el valor 1 para el domingo y 7 para el sábado. El filtro por `anio` usa la
partición de la tabla optimizada.

In [ ]:
por_dia = consulta(f"""
SELECT
  EXTRACT(DAYOFWEEK FROM starttime) AS dia_semana,
  COUNT(*)                          AS viajes,
  ROUND(AVG(tripduration), 1)       AS promedio_seg
FROM {TABLA}
WHERE anio >= 2016
  AND tripduration BETWEEN 60 AND 7200
GROUP BY dia_semana
ORDER BY dia_semana
""")
nombres = ["Dom", "Lun", "Mar", "Mié", "Jue", "Vie", "Sáb"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([nombres[d - 1] for d in por_dia.dia_semana], por_dia.viajes / 1e6)
ax.set_ylabel("Viajes (millones)")
plt.tight_layout()
plt.show()
por_dia

## 4. Comparación de los modelos

Sin un segundo argumento, `ML.EVALUATE` calcula las métricas sobre la porción de datos que BigQuery ML
reservó para evaluación durante el entrenamiento.

In [ ]:
comparacion = consulta(f"""
SELECT 'Iteración 1 (base)' AS modelo, mean_absolute_error, mean_squared_error,
       SQRT(mean_squared_error) AS rmse, median_absolute_error, r2_score
FROM ML.EVALUATE(MODEL {MODELO_BASE})
UNION ALL
SELECT 'Iteración 2 (filtrado)', mean_absolute_error, mean_squared_error,
       SQRT(mean_squared_error), median_absolute_error, r2_score
FROM ML.EVALUATE(MODEL {MODELO_V2})
""")
comparacion.round(4)

## 5. Inferencia sobre un viaje de ejemplo

Viaje de un abonado entre las estaciones 519 y 492, iniciado un lunes a las 8:00.

In [ ]:
consulta(f"""
SELECT predicted_tripduration,
       tripduration AS duracion_real,
       ABS(predicted_tripduration - tripduration) AS diferencia_absoluta
FROM ML.PREDICT(MODEL {MODELO_V2},
  (SELECT 1200 AS tripduration, '519' AS start_station_id, '492' AS end_station_id,
          'Subscriber' AS usertype, 8 AS hora_inicio, 2 AS dia_semana))
""")